# New signal

Copy this notebook, name it after the idea, and fill in the hypothesis before running anything.

**Hypothesis.** _What predicts what, and why should it (who is on the other side)?_
**Expected sign and horizon.** _e.g. a high value predicts a higher return over the next 5-20 days._
**Universe and period.** _e.g. the seven core equity indices, 1999 on._

Everything here is **exploratory**: recorded and counted, never a trial. When the idea is worth a real test, write
and lock a pre-registration and re-run it confirmatory (last section).

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from quantlab.lab import Lab
from quantlab.signals import library as lib
from quantlab.reporting import charts

charts.apply_style()
pd.set_option("display.width", 200)

lab = Lab(universe="equity_index_core", start="1999")
lab.instruments

## The signal

A function of `px` (`px.close`, `px.high`, `px.low`, `px.open`, `px.volume`, `px.returns`: dates x instruments).
Only data up to each date's close may be used: rolling windows, `.shift(n)` with n >= 0, and ranks *across* markets
(`axis=1`). The lab recomputes the signal on early data and warns if anything changed (it saw the future).

In [ ]:
idea = lab.signal(
    lambda px: (px.close.rolling(20).mean() / px.close.rolling(100).mean() - 1).rank(axis=1, pct=True) - 0.5,
    name="relative MA gap",
)
idea

## Test it

Signal card (does it predict?), strategy card (does trading it earn?), costs, and the book's signals for scale.

In [ ]:
report = lab.test(idea)

In [ ]:
report.strategy_card.table

In [ ]:
report.signal_card.table if report.signal_card is not None else 'no signal card'

## Against and inside the book

An idea that is weak alone can still be worth having if it is uncorrelated with what the book already holds.

In [ ]:
book = lab.compare_with_book(idea)

In [ ]:
combined = lab.combine(idea, "trend", overlay="h14_regime")

## Before pre-registering

How much history would a confirmatory test need for the effect you now expect? If the pass probability is low, the
test cannot answer its question: add markets or history, or expect a weaker claim.

In [ ]:
from quantlab.validation import power

years = len(lab.panel.index) / 252
expected_sharpe = 0.3      # be sceptical: the shrunk Sharpe on the card is a better guess than the backtest's
pd.Series(power.power_report(years, expected_sharpe, trials=lab.exploration_count() or 1))

## Going confirmatory

1. Write `research/prereg/H<n>.txt`: hypothesis, signal (exact code), universe, period, pass criteria (the strategy
   card's verdict), power. Then lock it: `from quantlab.validation import ledger; ledger.lock_prereg("H<n>")`.
2. Register it: `lab.store.register("H<n>", "title", status="pre-registered")`.
3. Re-run with `Lab(..., hypothesis="H<n>")`: tests now count as trials and the cards enter the lab-wide record.
4. If it passes, one look at the frozen holdout: `with lab.final_look(approved_by="owner"): lab.test(idea)`.